# InfectioGIT — Base de données SQLite (v2)

Ce notebook :
1. Crée le schéma SQLite complet (corrigé et étendu)
2. Détecte automatiquement la source d'un fichier JSON (BioModels / Zenodo / NCBI)
3. Parse chaque source avec ses propres règles d'extraction
4. Scanne le dépôt GitHub InfectioGIT et insère tout en base
5. Permet aussi un scan local (dev/test)

**Dépendances** : `sqlite3` (stdlib), `requests`, `pandas` (optionnel, pour visualiser)

In [73]:
import sqlite3
import json
import re
import requests
import pathlib
import pandas as pd
import base64 # pourquoi faire ?

---
## 1. Schéma de la base de données

Améliorations par rapport à la v1 :
- Clés primaires ajoutées sur **toutes** les tables (Maladie, Pathogene, Hote, Types_cellulaires)
- Champ `titre` et `source_db` ajoutés à `Modele`
- Table `Tissu` créée (était commentée)
- Table `Jeu_de_donnees` créée
- Tables de liaison `modele_hote` et `modele_jeu_de_donnees` ajoutées
- Bug corrigé : `Modèle` (avec accent) → `Modele` (cohérent partout)

In [74]:
DB_PATH = "infectio_git.db"

def create_database(db_path=DB_PATH):
    conn = sqlite3.connect(db_path)
    conn.execute("PRAGMA foreign_keys = ON")
    cur = conn.cursor()
    cur.executescript("""

    -- ══════════════════════════════════════════
    -- TABLES PRINCIPALES
    -- ══════════════════════════════════════════

    CREATE TABLE IF NOT EXISTS Modele (
        doi         TEXT PRIMARY KEY,  -- DOI ou identifiant stable (ex: BIOMD0000000958)
        titre       TEXT,
        formalisme  TEXT,              -- ODE, booléen, ABM, SBML, stochastique...
        echelle     TEXT,              -- moléculaire / cellulaire / tissulaire / hôte / épidémiologique
        annee       INTEGER,
        logiciel    TEXT,              -- COPASI, MATLAB, Python, R...
        source_db   TEXT               -- 'biomodels', 'zenodo', 'ncbi'
    );

    CREATE TABLE IF NOT EXISTS Maladie (
        doid_mondo  TEXT PRIMARY KEY,  -- ex: DOID:0080600
        nom         TEXT
    );

    CREATE TABLE IF NOT EXISTS Pathogene (
        identifiant_taxonomique TEXT PRIMARY KEY,  -- NCBI Taxonomy ID
        espece                  TEXT,
        classe                  TEXT,              -- virus / bacterie / parasite / champignon
        stade_cycle_vie         TEXT               -- réplicatif / latent / intracellulaire...
    );

    CREATE TABLE IF NOT EXISTS Hote (
        identifiant_taxonomique TEXT PRIMARY KEY,  -- NCBI Taxonomy ID
        espece                  TEXT,
        in_vitro_vivo           TEXT               -- 'in vitro' / 'in vivo' / 'in silico'
    );

    CREATE TABLE IF NOT EXISTS Tissu (
        identifiant_ontologie   TEXT PRIMARY KEY,  -- ex: UBERON:0002048
        nom                     TEXT               -- poumon, sang, intestin...
    );

    CREATE TABLE IF NOT EXISTS Types_cellulaires (
        ontologie_cellulaire    TEXT PRIMARY KEY,  -- ex: CL:0000235
        nom                     TEXT               -- macrophage, lymphocyte T...
    );

    CREATE TABLE IF NOT EXISTS Jeu_de_donnees (
        identifiant     TEXT PRIMARY KEY,  -- ex: GSE12345, accession clinique...
        source          TEXT,              -- GEO / clinique / in vitro / in vivo
        description     TEXT
    );

    -- ══════════════════════════════════════════
    -- TABLES DE LIAISON
    -- ══════════════════════════════════════════

    CREATE TABLE IF NOT EXISTS modele_maladie (
        modele_doi      TEXT,
        maladie_doid    TEXT,
        PRIMARY KEY (modele_doi, maladie_doid),
        FOREIGN KEY (modele_doi)   REFERENCES Modele(doi),
        FOREIGN KEY (maladie_doid) REFERENCES Maladie(doid_mondo)
    );

    CREATE TABLE IF NOT EXISTS modele_pathogene (
        modele_doi              TEXT,
        pathogene_taxonomie     TEXT,
        PRIMARY KEY (modele_doi, pathogene_taxonomie),
        FOREIGN KEY (modele_doi)            REFERENCES Modele(doi),
        FOREIGN KEY (pathogene_taxonomie)   REFERENCES Pathogene(identifiant_taxonomique)
    );

    CREATE TABLE IF NOT EXISTS modele_hote (
        modele_doi      TEXT,
        hote_taxonomie  TEXT,
        PRIMARY KEY (modele_doi, hote_taxonomie),
        FOREIGN KEY (modele_doi)    REFERENCES Modele(doi),
        FOREIGN KEY (hote_taxonomie) REFERENCES Hote(identifiant_taxonomique)
    );

    CREATE TABLE IF NOT EXISTS modele_tissu (
        modele_doi      TEXT,
        tissu_ontologie TEXT,
        PRIMARY KEY (modele_doi, tissu_ontologie),
        FOREIGN KEY (modele_doi)    REFERENCES Modele(doi),
        FOREIGN KEY (tissu_ontologie) REFERENCES Tissu(identifiant_ontologie)
    );

    CREATE TABLE IF NOT EXISTS modele_types_cellulaires (
        modele_doi          TEXT,
        typecell_ontologie  TEXT,
        PRIMARY KEY (modele_doi, typecell_ontologie),
        FOREIGN KEY (modele_doi)        REFERENCES Modele(doi),
        FOREIGN KEY (typecell_ontologie) REFERENCES Types_cellulaires(ontologie_cellulaire)
    );

    CREATE TABLE IF NOT EXISTS modele_jeu_de_donnees (
        modele_doi      TEXT,
        jdd_identifiant TEXT,
        PRIMARY KEY (modele_doi, jdd_identifiant),
        FOREIGN KEY (modele_doi)    REFERENCES Modele(doi),
        FOREIGN KEY (jdd_identifiant) REFERENCES Jeu_de_donnees(identifiant)
    );

    """)
    conn.commit()
    print(f"✅ Base de données créée / ouverte : {db_path}")
    return conn


conn = create_database()

✅ Base de données créée / ouverte : infectio_git.db


---
## 2. Détection automatique de la source

Chaque base de données a une structure JSON reconnaissable :

| Source | Clé distinctive |
|--------|----------------|
| BioModels | `modelLevelAnnotations`, `publicationId` |
| Zenodo | `conceptrecid`, URL zenodo.org dans `links` |
| NCBI | Liste avec `NlmUniqueID` ou `PmcRefCount` |

In [75]:
def detect_source(data):
    """Détecte la source (biomodels / zenodo / ncbi) depuis la structure JSON."""
    if isinstance(data, dict):
        if "modelLevelAnnotations" in data or "publicationId" in data or "submissionId" in data:
            return "biomodels"
        if "conceptrecid" in data or (
            "links" in data and "zenodo.org" in str(data.get("links", {}))
        ):
            return "zenodo"
    if isinstance(data, list) and len(data) > 0:
        item = data[0]
        if isinstance(item, dict) and (
            "NlmUniqueID" in item or "PmcRefCount" in item or "ArticleIds" in item
        ):
            return "ncbi"
    return "unknown"

---
## 3. Fonctions utilitaires

Inférence de classe de pathogène et de maladie à partir de texte libre.

In [76]:
# Taxons connus pour Homo sapiens
HUMAN_TAXON_IDS = {"9606"}

# Mots-clés → classe de pathogène
PATHOGEN_CLASS_KEYWORDS = {
    "virus": [
        "virus", "viral", "sars", "covid", "influenza", "dengue",
        "hiv", "ebola", "hcv", "hbv", "coronavirus", "rhinovirus",
        "adenovirus", "herpes", "rsv", "zika", "west nile"
    ],
    "bacterie": [
        "bacteria", "bacterial", "mycobacterium", "tuberculosis",
        "salmonella", "staphylococcus", "streptococcus", "escherichia",
        "klebsiella", "pseudomonas", "listeria", "clostridium"
    ],
    "parasite": [
        "parasite", "plasmodium", "malaria", "leishmania",
        "trypanosoma", "toxoplasma", "helminth", "giardia"
    ],
    "champignon": [
        "fungal", "fungi", "candida", "aspergillus", "cryptococcus"
    ],
}

# Mots-clés → DOID connus (pour inférence depuis texte libre)
DISEASE_KEYWORDS = {
    "covid": ("DOID:0080600", "COVID-19"),
    "sars-cov": ("DOID:0080600", "COVID-19"),
    "influenza": ("DOID:8469", "influenza"),
    "tuberculosis": ("DOID:399", "tuberculosis"),
    "malaria": ("DOID:12365", "malaria"),
    "dengue": ("DOID:12205", "dengue fever"),
    "hiv": ("DOID:526", "HIV infectious disease"),
    "hepatitis c": ("DOID:1883", "hepatitis C"),
    "hepatitis b": ("DOID:2043", "hepatitis B"),
    "ebola": ("DOID:4325", "Ebola hemorrhagic fever"),
    "zika": ("DOID:0060478", "Zika fever"),
}


def infer_pathogen_class(name: str) -> str:
    """Infère la classe d'un pathogène depuis son nom (NCBI Taxonomy)."""
    name_l = name.lower()
    for cls, keywords in PATHOGEN_CLASS_KEYWORDS.items():
        if any(k in name_l for k in keywords):
            return cls
    return ""


def infer_diseases_from_text(text: str) -> list:
    """Infère une liste de maladies (doid_mondo, nom) depuis du texte libre."""
    text_l = text.lower()
    found = []
    for keyword, (doid, nom) in DISEASE_KEYWORDS.items():
        if keyword in text_l:
            found.append({"doid_mondo": doid, "nom": nom})
    return found


def detect_logiciel_from_biomodels(data: dict) -> str:
    """Déduit le logiciel utilisé depuis les fichiers additionnels BioModels."""
    for f in data.get("files", {}).get("additional", []):
        name = f.get("name", "").lower()
        if ".cps" in name or "copasi" in name:
            return "COPASI"
        if "matlab" in name:
            return "MATLAB"
        if "octave" in name:
            return "Octave"
        if ".ode" in name:
            return "XPP/ODE"
    fmt = data.get("format", {}).get("name", "")
    return fmt if fmt else ""


print("✅ Utilitaires chargés")

✅ Utilitaires chargés


---
## 4. Parsers par source

Chaque parser retourne un dictionnaire normalisé :
```python
{
  "modele": {...},
  "maladies": [...],
  "pathogenes": [...],
  "hotes": [...],
  "tissus": [...],
  "types_cellulaires": [...],
  "jeux_de_donnees": [...]
}
```

In [77]:
def _empty_result():
    return {
        "modele": {},
        "maladies": [],
        "pathogenes": [],
        "hotes": [],
        "tissus": [],
        "types_cellulaires": [],
        "jeux_de_donnees": [],
    }


# ────────────────────────────────────────────
# 4a. PARSER BIOMODELS
# ────────────────────────────────────────────
def parse_biomodels(data: dict) -> dict:
    """
    Parse un fichier de métadonnées BioModels.
    Structure clé : modelLevelAnnotations (taxonomie NCBI, DOID, MAMO)
    """
    result = _empty_result()
    pub = data.get("publication", {})

    # Identifiant principal : publicationId (ex: BIOMD0000000958)
    # ou submissionId si publicationId absent
    identifier = data.get("publicationId") or data.get("submissionId", "")

    # Formalisme : depuis format.name + modellingApproach
    fmt_name = data.get("format", {}).get("name", "")       # ex: "SBML"
    fmt_ver  = data.get("format", {}).get("version", "")    # ex: "L2V4"
    approach = data.get("modellingApproach", {}).get("name", "")  # ex: "population model"
    formalisme = f"{fmt_name} {fmt_ver}".strip() if fmt_name else approach

    result["modele"] = {
        "doi":        identifier,
        "titre":      data.get("name", ""),
        "formalisme": formalisme,
        "echelle":    _infer_scale_biomodels(data),
        "annee":      pub.get("year"),
        "logiciel":   detect_logiciel_from_biomodels(data),
        "source_db":  "biomodels",
    }

    # ── Annotations sémantiques ──────────────────────────────────────
    for annot in data.get("modelLevelAnnotations", []):
        qualifier  = annot.get("qualifier", "")
        resource   = annot.get("resource", "")
        accession  = str(annot.get("accession", ""))
        name       = annot.get("name", "").strip()

        # Maladie : isVersionOf + ontologie de maladie
        if qualifier == "bqbiol:isVersionOf" and (
            "Disease" in resource or "doid" in resource.lower()
        ):
            doid = accession if accession.startswith("DOID:") else f"DOID:{accession}"
            result["maladies"].append({"doid_mondo": doid, "nom": name})

        # Taxon : séparer hôte (Homo sapiens) vs pathogène
        if qualifier == "bqbiol:hasTaxon":
            if accession in HUMAN_TAXON_IDS or "homo sapiens" in name.lower():
                result["hotes"].append({
                    "identifiant_taxonomique": accession,
                    "espece": name,
                    "in_vitro_vivo": "",
                })
            else:
                result["pathogenes"].append({
                    "identifiant_taxonomique": accession,
                    "espece": name,
                    "classe": infer_pathogen_class(name),
                    "stade_cycle_vie": "",
                })

    return result


def _infer_scale_biomodels(data: dict) -> str:
    """Infère l'échelle depuis le nom du modèle ou l'approche de modélisation."""
    approach = data.get("modellingApproach", {}).get("name", "").lower()
    name     = data.get("name", "").lower()
    text     = approach + " " + name
    if "population" in text or "epidemi" in text:
        return "épidémiologique"
    if "intracellular" in text or "molecular" in text:
        return "moléculaire"
    if "tissue" in text or "organ" in text:
        return "tissulaire"
    if "cellular" in text or "cell" in text:
        return "cellulaire"
    return ""


# ────────────────────────────────────────────
# 4b. PARSER ZENODO
# ────────────────────────────────────────────
def parse_zenodo(data: dict) -> dict:
    """
    Parse un fichier de métadonnées Zenodo.
    Les métadonnées biologiques structurées sont rares ici ;
    on infère maladie/pathogène depuis le titre/description.
    """
    result = _empty_result()
    meta = data.get("metadata", data)  # les champs sont parfois à la racine

    doi   = data.get("doi") or meta.get("doi", "")
    titre = data.get("title") or meta.get("title", "")

    # Langage de programmation → logiciel
    lang_list = meta.get("custom", {}).get("code:programmingLanguage", [])
    logiciel  = ", ".join(
        lang.get("title", {}).get("en", "") for lang in lang_list
    ) if lang_list else ""

    # Date de publication → année
    pub_date = meta.get("publication_date", "")
    annee = int(pub_date[:4]) if pub_date and pub_date[:4].isdigit() else None

    # Type de ressource → formalisme
    resource_type = meta.get("resource_type", {})
    formalisme = resource_type.get("title", resource_type.get("type", ""))

    result["modele"] = {
        "doi":        doi,
        "titre":      titre,
        "formalisme": formalisme,
        "echelle":    "",  # non présent dans Zenodo
        "annee":      annee,
        "logiciel":   logiciel,
        "source_db":  "zenodo",
    }

    # Inférence depuis texte libre (titre + description)
    desc = meta.get("description", "")
    text = titre + " " + desc
    result["maladies"] = infer_diseases_from_text(text)

    # Jeux de données liés (related_identifiers)
    for rel in meta.get("related_identifiers", []):
        ident = rel.get("identifier", "")
        if ident:
            result["jeux_de_donnees"].append({
                "identifiant": ident,
                "source":      rel.get("resource_type", ""),
                "description": f"relation: {rel.get('relation', '')}",
            })

    return result


# ────────────────────────────────────────────
# 4c. PARSER NCBI
# ────────────────────────────────────────────
def parse_ncbi(data) -> dict:
    """
    Parse un fichier de métadonnées NCBI/PubMed (format liste).
    Principalement des métadonnées de publication ; peu de données biologiques.
    Inférence maladie/pathogène depuis le titre de l'article.
    """
    result = _empty_result()
    item = data[0] if isinstance(data, list) else data

    doi = (
        item.get("DOI")
        or item.get("ArticleIds", {}).get("doi", "")
        or item.get("ELocationID", "").replace("doi: ", "")
    )
    titre = item.get("Title", "")

    # Année
    pub_date = item.get("PubDate", "")
    annee = int(pub_date[:4]) if pub_date and pub_date[:4].isdigit() else None

    result["modele"] = {
        "doi":        doi,
        "titre":      titre,
        "formalisme": "",  # non structuré dans NCBI
        "echelle":    "",
        "annee":      annee,
        "logiciel":   "",
        "source_db":  "ncbi",
    }

    # Inférence maladie depuis titre
    result["maladies"] = infer_diseases_from_text(titre)

    return result


# ────────────────────────────────────────────
# Dispatcher principal
# ────────────────────────────────────────────
def parse_metadata(data) -> dict:
    """Détecte la source et appelle le bon parser."""
    source = detect_source(data)
    if source == "biomodels":
        return parse_biomodels(data)
    elif source == "zenodo":
        return parse_zenodo(data)
    elif source == "ncbi":
        return parse_ncbi(data)
    else:
        print("  ⚠️  Source inconnue, fichier ignoré.")
        return _empty_result()


print("✅ Parsers chargés (BioModels / Zenodo / NCBI)")

✅ Parsers chargés (BioModels / Zenodo / NCBI)


---
## 5. Insertion en base de données

In [78]:
def insert_parsed(conn: sqlite3.Connection, parsed: dict, verbose: bool = True):
    """Insère un enregistrement parsé dans la base. INSERT OR IGNORE = pas de doublon."""
    m = parsed["modele"]
    doi = m.get("doi", "").strip()

    if not doi:
        print("  ⚠️  Pas d'identifiant (DOI) — enregistrement ignoré")
        return False

    cur = conn.cursor()

    # ── Modele ────────────────────────────────────────────────────────
    cur.execute(
        "INSERT OR IGNORE INTO Modele (doi, titre, formalisme, echelle, annee, logiciel, source_db) "
        "VALUES (?, ?, ?, ?, ?, ?, ?)",
        (doi, m.get("titre"), m.get("formalisme"), m.get("echelle"),
         m.get("annee"), m.get("logiciel"), m.get("source_db"))
    )

    # ── Maladies ──────────────────────────────────────────────────────
    for d in parsed["maladies"]:
        cur.execute("INSERT OR IGNORE INTO Maladie VALUES (?, ?)",
                    (d["doid_mondo"], d["nom"]))
        cur.execute("INSERT OR IGNORE INTO modele_maladie VALUES (?, ?)",
                    (doi, d["doid_mondo"]))

    # ── Pathogènes ────────────────────────────────────────────────────
    for p in parsed["pathogenes"]:
        cur.execute("INSERT OR IGNORE INTO Pathogene VALUES (?, ?, ?, ?)",
                    (p["identifiant_taxonomique"], p["espece"],
                     p.get("classe", ""), p.get("stade_cycle_vie", "")))
        cur.execute("INSERT OR IGNORE INTO modele_pathogene VALUES (?, ?)",
                    (doi, p["identifiant_taxonomique"]))

    # ── Hôtes ─────────────────────────────────────────────────────────
    for h in parsed["hotes"]:
        cur.execute("INSERT OR IGNORE INTO Hote VALUES (?, ?, ?)",
                    (h["identifiant_taxonomique"], h["espece"],
                     h.get("in_vitro_vivo", "")))
        cur.execute("INSERT OR IGNORE INTO modele_hote VALUES (?, ?)",
                    (doi, h["identifiant_taxonomique"]))

    # ── Tissus ────────────────────────────────────────────────────────
    for t in parsed["tissus"]:
        cur.execute("INSERT OR IGNORE INTO Tissu VALUES (?, ?)",
                    (t["identifiant_ontologie"], t["nom"]))
        cur.execute("INSERT OR IGNORE INTO modele_tissu VALUES (?, ?)",
                    (doi, t["identifiant_ontologie"]))

    # ── Types cellulaires ─────────────────────────────────────────────
    for tc in parsed["types_cellulaires"]:
        cur.execute("INSERT OR IGNORE INTO Types_cellulaires VALUES (?, ?)",
                    (tc["ontologie_cellulaire"], tc.get("nom", "")))
        cur.execute("INSERT OR IGNORE INTO modele_types_cellulaires VALUES (?, ?)",
                    (doi, tc["ontologie_cellulaire"]))

    # ── Jeux de données ───────────────────────────────────────────────
    for jdd in parsed["jeux_de_donnees"]:
        cur.execute("INSERT OR IGNORE INTO Jeu_de_donnees VALUES (?, ?, ?)",
                    (jdd["identifiant"], jdd.get("source", ""), jdd.get("description", "")))
        cur.execute("INSERT OR IGNORE INTO modele_jeu_de_donnees VALUES (?, ?)",
                    (doi, jdd["identifiant"]))

    conn.commit()
    if verbose:
        print(f"  ✅  Inséré : {doi}")
    return True


print("✅ Fonction d'insertion chargée")

✅ Fonction d'insertion chargée


---
## 6. Scanner le dépôt GitHub InfectioGIT

Utilise l'API GitHub pour lister tous les `.json` dans `/metadata/`,
puis télécharge et insère chacun.

> **Note** : pour les dépôts privés ou en cas de rate-limit, ajouter un token GitHub :
> `headers["Authorization"] = "token VOTRE_TOKEN"`

In [ ]:
GITHUB_REPO   = "thalieh/InfectioGIT"
GITHUB_BRANCH = "main"
METADATA_FOLDER = "Results"  # dossier à scanner dans le repo

# Optionnel : token GitHub pour éviter le rate-limit (60 req/h sans token)
GITHUB_TOKEN = "TOKEN_ICI"  # laisser vide pour accès public


def _github_headers():
    h = {"Accept": "application/vnd.github+json"}
    if GITHUB_TOKEN:
        h["Authorization"] = f"token {GITHUB_TOKEN}"
    return h


def list_json_files_github(
    repo=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    folder=METADATA_FOLDER,
) -> list:
    """
    Liste tous les fichiers .json dans `folder/` via l'API GitHub.
    Retourne une liste de chemins relatifs (ex: 'metadata/biomodels/BIOMD0000000958.json').
    """
    url = f"https://api.github.com/repos/{repo}/git/trees/{branch}?recursive=1"
    resp = requests.get(url, headers=_github_headers(), timeout=30)

    if resp.status_code == 403:
        raise RuntimeError(
            "Rate-limit GitHub atteint. Attendez ou ajoutez un GITHUB_TOKEN."
        )
    if resp.status_code != 200:
        raise RuntimeError(f"Erreur API GitHub {resp.status_code}: {resp.text[:200]}")

    tree = resp.json().get("tree", [])
    json_files = [
        item["path"]
        for item in tree
        if item["type"] == "blob"
        and item["path"].startswith(folder)
        and item["path"].endswith(".json")
    ]
    return json_files


def fetch_json_github(path: str, repo=GITHUB_REPO, branch=GITHUB_BRANCH):
    """Télécharge le JSON. Pour les dépôts publics, cette URL résout le LFS d'elle-même."""
    # On utilise l'URL raw de github.com (pas l'API) pour laisser GitHub gérer le LFS
    url = f"https://github.com/{repo}/raw/{branch}/{path}"
    resp = requests.get(url, timeout=30)
    
    if resp.status_code != 200:
        print(f"  ❌ Erreur {resp.status_code} pour {path}")
        return None
        
    try:
        return resp.json()
    except json.JSONDecodeError:
        # Si ça échoue encore, c'est que GitHub renvoie quand même le pointeur LFS (texte)
        if "git-lfs" in resp.text:
            print(f"  ⚠️ LFS bloque toujours sur : {path}")
        return None

def scan_github_and_populate(
    conn: sqlite3.Connection,
    repo=GITHUB_REPO,
    branch=GITHUB_BRANCH,
    folder=METADATA_FOLDER,
):
    """
    Pipeline complet :
    1. Liste les JSON dans le repo GitHub
    2. Télécharge chaque fichier
    3. Détecte la source et parse
    4. Insère en base
    """
    print(f"🔍 Scan de {repo}/{folder} (branche: {branch})...")
    json_files = list_json_files_github(repo, branch, folder)
    print(f"   {len(json_files)} fichier(s) JSON trouvé(s)\n")

    succes, echecs = 0, 0

    for path in json_files:
        print(f"📄 {path}")
        data = fetch_json_github(path, repo, branch)
        if data is None:
            echecs += 1
            continue

        source = detect_source(data)
        print(f"   source: {source}")

        if source == "unknown":
            print("   ⚠️  Source inconnue, ignoré.")
            echecs += 1
            continue

        parsed = parse_metadata(data)
        ok = insert_parsed(conn, parsed)
        if ok:
            succes += 1
        else:
            echecs += 1

    print(f"\n🎉 Terminé — {succes} insertions, {echecs} erreurs/ignorés")


print("✅ Scanner GitHub chargé")

✅ Scanner GitHub chargé


---
## 7. Scanner un dossier local (dev / tests)

Utile pour tester avec les 3 exemples JSON fournis avant de lancer le scan GitHub.

In [80]:
def scan_local_and_populate(conn: sqlite3.Connection, folder_path: str = "metadata"):
    """
    Parcourt récursivement un dossier local à la recherche de .json,
    puis insère chaque fichier en base.
    """
    folder = pathlib.Path(folder_path)
    if not folder.exists():
        print(f"❌ Dossier introuvable : {folder_path}")
        return

    json_files = sorted(folder.rglob("*.json"))
    print(f"🔍 {len(json_files)} fichier(s) JSON dans {folder_path}\n")

    succes, echecs = 0, 0

    for path in json_files:
        print(f"📄 {path}")
        try:
            with open(path, "r", encoding="utf-8") as f:
                data = json.load(f)
        except (json.JSONDecodeError, OSError) as e:
            print(f"  ❌ Erreur lecture : {e}")
            echecs += 1
            continue

        source = detect_source(data)
        print(f"   source: {source}")
        parsed = parse_metadata(data)
        ok = insert_parsed(conn, parsed)
        if ok:
            succes += 1
        else:
            echecs += 1

    print(f"\n🎉 Terminé — {succes} insertions, {echecs} erreurs/ignorés")


print("✅ Scanner local chargé")

✅ Scanner local chargé


---
## 8. LANCEMENT

Choisis le mode qui te convient :

In [81]:
# ══ OPTION A : scan du dépôt GitHub InfectioGIT ════════════════════════
scan_github_and_populate(conn)

# ══ OPTION B : test local avec les 3 fichiers d'exemple ════════════════
#scan_local_and_populate(conn, folder_path="examples")  # mettre tes JSON dans examples/

# ══ OPTION C : insertion manuelle d'un seul fichier ════════════════════
# with open("metadata_ex3_biomodel.json") as f:
#     data = json.load(f)
# parsed = parse_metadata(data)
# insert_parsed(conn, parsed)

🔍 Scan de thalieh/InfectioGIT/Results (branche: main)...
   281 fichier(s) JSON trouvé(s)

📄 Results/Models/Chikungunya/Zenodo/DOI15100955/metadata/metadata_15100955.json
   source: zenodo
  ✅  Inséré : 10.5281/zenodo.15100955
📄 Results/Models/Covid/Biomodels/curated/ordinary_differential_equation_model/BIOMD0000000979/metadata/BIOMD0000000979_web_metadata.json
   source: biomodels
  ✅  Inséré : BIOMD0000000979
📄 Results/Models/Covid/Biomodels/curated/ordinary_differential_equation_model/BIOMD0000000979/metadata/files_list.json
   source: unknown
   ⚠️  Source inconnue, ignoré.
📄 Results/Models/Covid/Biomodels/curated/population_model/BIOMD0000000955/metadata/BIOMD0000000955_web_metadata.json
   source: biomodels
  ✅  Inséré : BIOMD0000000955
📄 Results/Models/Covid/Biomodels/curated/population_model/BIOMD0000000955/metadata/files_list.json
   source: unknown
   ⚠️  Source inconnue, ignoré.
📄 Results/Models/Covid/Biomodels/curated/population_model/BIOMD0000000956/metadata/BIOMD000000095

---
## 9. Vérification et visualisation de la base

In [82]:
# Résumé du contenu de la base
tables = [
    "Modele", "Maladie", "Pathogene", "Hote",
    "Tissu", "Types_cellulaires", "Jeu_de_donnees",
    "modele_maladie", "modele_pathogene", "modele_hote",
    "modele_tissu", "modele_types_cellulaires", "modele_jeu_de_donnees",
]

print("📊 Contenu de la base de données\n" + "=" * 40)
for t in tables:
    n = conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    print(f"  {t:<35} : {n:>5} ligne(s)")

📊 Contenu de la base de données
  Modele                              :   129 ligne(s)
  Maladie                             :     6 ligne(s)
  Pathogene                           :    19 ligne(s)
  Hote                                :     1 ligne(s)
  Tissu                               :     0 ligne(s)
  Types_cellulaires                   :     0 ligne(s)
  Jeu_de_donnees                      :     8 ligne(s)
  modele_maladie                      :    26 ligne(s)
  modele_pathogene                    :    59 ligne(s)
  modele_hote                         :    64 ligne(s)
  modele_tissu                        :     0 ligne(s)
  modele_types_cellulaires            :     0 ligne(s)
  modele_jeu_de_donnees               :     8 ligne(s)


In [83]:
# Aperçu des modèles
pd.read_sql_query("SELECT * FROM Modele", conn)

,doi,titre,formalisme,echelle,annee,logiciel,source_db
0,10.5281/zenodo.16743227,Longitudinal antibody profiling after dengue r...,Computational notebook,,2025.0,R,zenodo
1,BIOMD0000000958,Ndairou2020 - early-stage transmission dynamic...,SBML L2V4,épidémiologique,2020.0,MATLAB,biomodels
2,10.3389/fgene.2019.00633,FindTargetsWEB: A User-Friendly Tool for Ident...,,,2019.0,,ncbi
3,10.5281/zenodo.15100955,Risk assessment and perspectives of local tran...,Software,,2025.0,,zenodo
4,BIOMD0000000979,Malkov2020 - SEIRS model of COVID-19 transmiss...,SBML L3V1,,2020.0,MATLAB,biomodels
...,...,...,...,...,...,...,...
124,MODEL1011090002,Bordbar2010_M_tuberculosis_Macrophage,SBML L2V4,,2010.0,SBML,biomodels
125,MODEL1411110000,Rienksma2014 - Genome-scale constraint-based m...,SBML L3V1,,2014.0,SBML,biomodels
126,MODEL1507180018,Fang2010 - Genome-scale metabolic network of M...,SBML L3V1,,2010.0,SBML,biomodels
127,MODEL1507180021,Beste2007 - Genome-scale metabolic network of ...,SBML L3V1,,2007.0,SBML,biomodels


In [84]:
# Jointure : modèles × maladies
pd.read_sql_query("""
    SELECT m.titre, m.formalisme, m.annee, m.source_db,
           ma.nom AS maladie, ma.doid_mondo
    FROM Modele m
    JOIN modele_maladie mm ON m.doi = mm.modele_doi
    JOIN Maladie ma        ON mm.maladie_doid = ma.doid_mondo
""", conn)

,titre,formalisme,annee,source_db,maladie,doid_mondo
0,Longitudinal antibody profiling after dengue r...,Computational notebook,2025,zenodo,dengue fever,DOID:12205
1,Ndairou2020 - early-stage transmission dynamic...,SBML L2V4,2020,biomodels,COVID-19,DOID:0080600
2,Risk assessment and perspectives of local tran...,Software,2025,zenodo,dengue fever,DOID:12205
3,Roda2020 - SIR model of COVID-19 spread in Wuhan,SBML L2V4,2020,biomodels,COVID-19,DOID:0080600
4,Paiva2020 - SEIAHRD model of transmission dyna...,SBML L2V4,2020,biomodels,COVID-19,DOID:0080600
5,Paiva2020 - SEIAHRD model of transmission dyna...,SBML L2V4,2020,biomodels,,DOID:0000503
6,Zhao2020 - SUQC model of COVID-19 transmission...,SBML L2V4,2020,biomodels,COVID-19,DOID:0080600
7,Weitz2020 - SIR model of COVID-19 transmission...,SBML L2V4,2020,biomodels,COVID-19,DOID:0080600
8,Weitz2020 - SIR model of COVID-19 transmission...,SBML L2V4,2020,biomodels,,DOID:0000503
9,Mwalili2020 - SEIR model of COVID-19 transmiss...,SBML L3V1,2020,biomodels,,DOID:0000503


In [85]:
# Jointure : modèles × pathogènes × hôtes
pd.read_sql_query("""
    SELECT m.titre, m.annee,
           p.espece AS pathogene, p.classe,
           h.espece AS hote
    FROM Modele m
    LEFT JOIN modele_pathogene mp ON m.doi = mp.modele_doi
    LEFT JOIN Pathogene p          ON mp.pathogene_taxonomie = p.identifiant_taxonomique
    LEFT JOIN modele_hote mh       ON m.doi = mh.modele_doi
    LEFT JOIN Hote h               ON mh.hote_taxonomie = h.identifiant_taxonomique
""", conn)

,titre,annee,pathogene,classe,hote
0,Longitudinal antibody profiling after dengue r...,2025.0,None,None,None
1,Ndairou2020 - early-stage transmission dynamic...,2020.0,Severe acute respiratory syndrome coronavirus 2,virus,Homo sapiens
2,FindTargetsWEB: A User-Friendly Tool for Ident...,2019.0,None,None,None
3,Risk assessment and perspectives of local tran...,2025.0,None,None,None
4,Malkov2020 - SEIRS model of COVID-19 transmiss...,2020.0,Severe acute respiratory syndrome coronavirus 2,virus,Homo sapiens
...,...,...,...,...,...
130,Bordbar2010_M_tuberculosis_Macrophage,2010.0,Mycobacterium tuberculosis,bacterie,Homo sapiens
131,Rienksma2014 - Genome-scale constraint-based m...,2014.0,None,None,None
132,Fang2010 - Genome-scale metabolic network of M...,2010.0,None,None,None
133,Beste2007 - Genome-scale metabolic network of ...,2007.0,None,None,None


In [87]:
# Fermer la connexion proprement
conn.close()
print("🔒 Connexion fermée")

🔒 Connexion fermée
